In [3]:
"""
GLO STEP 1 - Build lake-change tables from the Glacial Lake Observatory (Nepal)
==============================================================================

Replaces the synthetic Step 1. Instead of generating lakes, it reads REAL lakes.

Data: Rawlins, Watson, Bhambri, Khadka & Chand (2026), "Glacial Lake Observatory
(GLO): annual dataset of glacial lakes in Nepal and transboundary catchments
(2017-2024)", Earth System Science Data 18, 5143. Zenodo 10.5281/zenodo.17802334
(CC BY 4.0). File used: S2_20172024_TB_GLOID_v1_0.gpkg (yearly Sentinel-2
outlines, one row per lake per year, linked by GLO_ID).

Outputs (same roles as the synthetic Step 1):
  data/train_lakes.csv          2018 -> 2021 change, inputs from 2018   (Step 2)
  data/analysis_features.csv    inputs from 2021, NO answers            (Step 3)
  data/analysis_answer_key.csv  observed 2021 -> 2024 change            (Step 3, validation only)
  data/generation_meta.json

Inputs available in GLO (all measured at the START year t0):
  log area, relative area uncertainty, elevation (ALOS median), compactness
  (from area and perimeter), glacier-fed flag, number of polygons, and the
  previous year's change (lag: t0-1 -> t0). GLO has no glacier area, slope,
  debris cover, mass balance or dam type.

Target: y = ln(A_t1 / A_t0), the 3-year log area change.

Run:  python glo_step1_build_tables.py [--gpkg path/to/S2_20172024_TB_GLOID_v1_0.gpkg]
In a notebook: paste the cell (runs with defaults) or call main(["--gpkg", r"C:\full\path\file.gpkg"]).
Without --gpkg it searches: the current folder, ./data, ~/Downloads/glacial_lake_data, ~/Downloads.
"""
from __future__ import annotations

import argparse
import json
import os
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR, FIG_DIR = "data", "figures"
FEATURES = ["log_area_t0", "rel_area_error", "elevation_m", "compactness", "glacier_fed",
            "n_polygons_t0", "lag_change", "lag_missing"]


GPKG_NAME = "S2_20172024_TB_GLOID_v1_0.gpkg"


def _is_valid_gpkg(p: Path) -> bool:
    """True if p is a real (non-empty) SQLite/GeoPackage file."""
    try:
        if not p.is_file() or p.stat().st_size < 1024:
            return False
        with open(p, "rb") as fh:
            return fh.read(16).startswith(b"SQLite format 3")
    except OSError:
        return False


def find_gpkg(path: str | None) -> Path:
    """Find the GLO GeoPackage. Tries the given path first, then common locations.
    Skips empty files (a failed earlier run can leave a 0-byte file behind)."""
    name = Path(path).name if path else GPKG_NAME
    here = Path.cwd()
    candidates = []
    if path:
        candidates.append(Path(path).expanduser())
    candidates += [here / name, here / "data" / name,
                   Path.home() / "Downloads" / "glacial_lake_data" / name,
                   Path.home() / "Downloads" / name,
                   Path.home() / "OneDrive" / "Downloads" / "glacial_lake_data" / name]
    try:  # folder of this script, when run as a .py file
        candidates.append(Path(__file__).resolve().parent / "data" / name)
    except NameError:
        pass
    for c in candidates:
        if _is_valid_gpkg(c):
            return c.resolve()
    empty = [str(c) for c in candidates if c.is_file() and not _is_valid_gpkg(c)]
    msg = (f"Could not find a valid '{name}'.\nLooked in:\n  " + "\n  ".join(str(c) for c in candidates) +
           "\nDownload it from https://zenodo.org/records/17802334 and either put it in one of those "
           "places or pass its full path, e.g. main(['--gpkg', r'C:\\path\\to\\" + name + "']).")
    if empty:
        msg += "\nThese files exist but are empty or not a GeoPackage (safe to delete):\n  " + "\n  ".join(empty)
    raise FileNotFoundError(msg)


def read_glo(path: str | None) -> pd.DataFrame:
    """Read the attribute table of the GeoPackage (no GIS library needed).
    Opens the file read-only, so a wrong path can never create an empty file."""
    gpkg = find_gpkg(path)
    print(f"  Reading {gpkg}")
    con = sqlite3.connect(gpkg.as_uri() + "?mode=ro", uri=True)
    table = con.execute("select table_name from gpkg_contents where data_type='features'").fetchone()[0]
    cols = [r[1] for r in con.execute(f"pragma table_info('{table}')") if r[1] not in ("geom", "fid")]
    df = pd.read_sql(f"select {', '.join(cols)} from '{table}'", con)
    con.close()
    return df


def lake_year_table(df: pd.DataFrame) -> pd.DataFrame:
    """One row per lake per year (a few lakes have several polygons in a year)."""
    df = df.dropna(subset=["AREA_YEAR"]).copy()
    df["AREA_YEAR"] = df.AREA_YEAR.astype(int)
    g = df.groupby(["GLO_ID", "AREA_YEAR"])
    out = g.agg(area=("AREA", "sum"), perimeter=("PERIMETER", "sum"),
                n_polygons=("AREA", "size"), elevation=("ELEVATION_MEDIAN", "median"),
                lon=("LONGITUDE", "mean"), lat=("LATITUDE", "mean"),
                basin=("BASIN", "first"), country=("COUNTRY", "first"),
                connectivity=("CONNECTIVITY", lambda s: s.mode().iat[0] if s.notna().any() else np.nan),
                ts_outlier=("TS_OUTLIER", lambda s: (s == "TRUE").any()))
    out["area_unc"] = g.AREA_UNCERTAINTY.apply(lambda s: float(np.sqrt((s.fillna(0) ** 2).sum())))
    return out.reset_index()


def period_table(ly: pd.DataFrame, t0: int, t1: int | None):
    wide = {y: ly[ly.AREA_YEAR == y].set_index("GLO_ID") for y in sorted(ly.AREA_YEAR.unique())}
    a0 = wide[t0]
    lag = wide.get(t0 - 1)
    rows = pd.DataFrame(index=a0.index)
    rows["lake_id"] = a0.index
    rows["lat"], rows["lon"] = a0.lat, a0.lon
    rows["basin"], rows["country"] = a0.basin, a0.country
    rows["connectivity"] = a0.connectivity.fillna("Unknown")
    rows["area_t0_km2"] = a0.area
    rows["log_area_t0"] = np.log(a0.area)
    rows["rel_area_error"] = a0.area_unc / a0.area
    rows["elevation_m"] = a0.elevation
    rows["compactness"] = 4 * np.pi * a0.area / a0.perimeter ** 2      # km² and km -> unitless
    rows["glacier_fed"] = (a0.connectivity == "Glacier-fed").astype(float)
    rows["n_polygons_t0"] = a0.n_polygons
    if lag is not None:
        la = lag.area.reindex(rows.index)
        rows["lag_change"] = np.log(a0.area / la)
    else:
        rows["lag_change"] = np.nan
    rows["lag_missing"] = rows.lag_change.isna().astype(float)
    rows["ts_outlier_t0"] = a0.ts_outlier
    counts = {"present_t0": int(len(rows))}
    if t1 is not None:
        a1 = wide[t1].area
        both = rows.index.intersection(a1.index)
        counts.update({"present_both": int(len(both)), "vanished_by_t1": int(len(rows) - len(both)),
                       "appeared_by_t1": int(len(a1.index.difference(rows.index)))})
        rows["area_t1_km2"] = a1.reindex(rows.index)
        rows["log_area_change"] = np.log(rows.area_t1_km2 / rows.area_t0_km2)
    return rows.reset_index(drop=True), counts


def overview_figure(tr, an, key, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.spines.top": False,
                         "axes.spines.right": False, "axes.edgecolor": "#c3c2b7"})
    A, B = "#2a78d6", "#eb6834"
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.2), facecolor="#fcfcfb")
    ax = axs[0]
    ax.scatter(an.lon, an.lat, s=3, c="#c3c2b7", lw=0, label=f"lakes in 2021 (n={len(an):,})")
    gf = an.glacier_fed == 1
    ax.scatter(an.lon[gf], an.lat[gf], s=3, c=A, lw=0, label="glacier-fed")
    ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)"); ax.legend(frameon=False, fontsize=8, markerscale=3)
    ax.set_title("GLO lakes: Nepal and transboundary catchments", loc="left")
    ax = axs[1]
    bins = np.linspace(-2.5, 2.5, 60)
    ax.hist(tr.log_area_change.clip(-2.5, 2.5), bins, density=True, alpha=0.6, color=A, label="2018→2021 (train)")
    k = key.log_area_change.dropna()
    ax.hist(k.clip(-2.5, 2.5), bins, density=True, alpha=0.6, color=B, label="2021→2024 (analysis)")
    ax.set_xlabel("3-year log area change"); ax.set_yticks([]); ax.legend(frameon=False, fontsize=8)
    ax.set_title("Observed change in the two periods", loc="left")
    ax = axs[2]
    ax.hist(np.log10(tr.area_t0_km2), 50, alpha=0.6, color=A, density=True, label="2018")
    ax.hist(np.log10(an.area_t0_km2), 50, alpha=0.6, color=B, density=True, label="2021")
    ax.set_xlabel("log10 lake area (km²)"); ax.set_yticks([]); ax.legend(frameon=False, fontsize=8)
    ax.set_title("Lake sizes at the start of each period", loc="left")
    fig.tight_layout(); fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--gpkg", default=None,
                    help="path to S2_20172024_TB_GLOID_v1_0.gpkg (default: search ./, ./data, "
                         "~/Downloads/glacial_lake_data, ~/Downloads)")
    ap.add_argument("--train-start", type=int, default=2018)
    ap.add_argument("--horizon", type=int, default=3)
    ap.add_argument("--analysis-start", type=int, default=2021)
    args = ap.parse_known_args(argv)[0]           # notebook-safe: ignores Jupyter's -f argument
    os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

    raw = read_glo(args.gpkg)
    ly = lake_year_table(raw)
    t0, t1 = args.train_start, args.train_start + args.horizon
    a0, a1 = args.analysis_start, args.analysis_start + args.horizon
    if a0 < t1:
        raise SystemExit("Analysis period must start no earlier than the end of the training period.")
    train, ctr = period_table(ly, t0, t1)
    train = train.dropna(subset=["log_area_change"]).reset_index(drop=True)
    ana, can = period_table(ly, a0, a1)
    key = ana[["lake_id", "area_t1_km2", "log_area_change"]].copy()
    feats = ana.drop(columns=["area_t1_km2", "log_area_change"])

    train.to_csv(f"{DATA_DIR}/train_lakes.csv", index=False)
    feats.to_csv(f"{DATA_DIR}/analysis_features.csv", index=False)
    key.to_csv(f"{DATA_DIR}/analysis_answer_key.csv", index=False)
    meta = {"source": "Glacial Lake Observatory (Rawlins et al. 2026), Zenodo 10.5281/zenodo.17802334, CC BY 4.0",
            "file": GPKG_NAME if args.gpkg is None else os.path.basename(args.gpkg), "outlines": int(len(raw)), "lakes": int(raw.GLO_ID.nunique()),
            "train_period": [t0, t1], "analysis_period": [a0, a1], "train_counts": ctr, "analysis_counts": can,
            "features": FEATURES,
            "bbox": [float(ly.lat.min()), float(ly.lat.max()), float(ly.lon.min()), float(ly.lon.max())]}
    json.dump(meta, open(f"{DATA_DIR}/generation_meta.json", "w"), indent=2)
    overview_figure(train, feats, key, f"{FIG_DIR}/glo_step1_overview.png")

    print("GLO STEP 1 complete")
    print(f"  {meta['outlines']:,} outlines, {meta['lakes']:,} lakes")
    print(f"  Training  {t0}->{t1}: {ctr}")
    print(f"  Analysis  {a0}->{a1}: {can}  (answers stored separately)")
    print(f"  Median 3-yr change: train {train.log_area_change.median():+.3f}, "
          f"analysis {key.log_area_change.median():+.3f}")


if __name__ == "__main__":
    main()

<>:28: SyntaxWarning: invalid escape sequence '\p'
<>:28: SyntaxWarning: invalid escape sequence '\p'
C:\Users\goone\AppData\Local\Temp\ipykernel_14648\319841099.py:28: SyntaxWarning: invalid escape sequence '\p'
  In a notebook: paste the cell (runs with defaults) or call main(["--gpkg", r"C:\full\path\file.gpkg"]).


  Reading C:\Users\goone\Downloads\glacial_lake_data\S2_20172024_TB_GLOID_v1_0.gpkg
GLO STEP 1 complete
  22,417 outlines, 4,152 lakes
  Training  2018->2021: {'present_t0': 2633, 'present_both': 2274, 'vanished_by_t1': 359, 'appeared_by_t1': 481}
  Analysis  2021->2024: {'present_t0': 2755, 'present_both': 2429, 'vanished_by_t1': 326, 'appeared_by_t1': 725}  (answers stored separately)
  Median 3-yr change: train +0.028, analysis +0.010


In [4]:
"""
GLO STEP 2 - Train the lake-change model and calibrate split conformal intervals
================================================================================

Same procedure as the synthetic Step 2, on REAL lakes (GLO, 2018 -> 2021):
1. Spatial blocking: 0.5° cells; whole cells go to train 60 % / calibration 20 % / test 20 %.
2. Point model f(x): histogram gradient boosting on GLO inputs (lat/lon are NOT inputs).
3. Difficulty model sigma(x): booster on spatially out-of-fold absolute residuals.
4. Normalized split conformal prediction, score |y - f(x)| / sigma(x),
   training-conditional rank (delta = 0.05: >= 90 % coverage with 95 % confidence),
   pooled and per lake connectivity (glacier-fed / non glacier-fed = "Mondrian").
   Also CQR (conformalized quantile regression; Romano et al. 2019): two quantile
   models (5 % and 95 %) give a band, score max(q_lo - y, y - q_hi), interval
   [q_lo - q, q_hi + q]; intervals can be lopsided.
5. Checks on the untouched test blocks + 300 re-drawn calibration/test splits.
6. Method choice WITHOUT looking at 2021-2024: normalized-per-group vs
   CQR-per-group are compared on the held-out 2018-2021 test blocks; the one
   with the lower interval score (coverage >= 0.88 required) is saved as
   "preferred_method" and used by Step 3 by default.

Input : data/train_lakes.csv (from glo_step1_build_tables.py)
Output: models/glo_cp_bundle.joblib, outputs/glo_step2_metrics.json,
        figures/glo_step2_training_diagnostics.png

Run:  python glo_step2_train_calibrate.py [--alpha 0.1] [--block-deg 0.5] [--delta 0.05]
In a notebook: paste the cell (runs with defaults) or call main(["--alpha", "0.05"]).
"""
from __future__ import annotations

import argparse
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold

FEATURES = ["log_area_t0", "rel_area_error", "elevation_m", "compactness", "glacier_fed",
            "n_polygons_t0", "lag_change", "lag_missing"]
TARGET = "log_area_change"
GROUP_COL = "connectivity"


# ---------------------------------------------------------------------------
# Shared helpers (Step 3 imports these)
# ---------------------------------------------------------------------------
def make_X(df: pd.DataFrame) -> pd.DataFrame:
    return df[FEATURES].astype(float)


def spatial_blocks(lat, lon, deg):
    return (np.floor(np.asarray(lat, float) / deg).astype(int).astype(str) + "_"
            + np.floor(np.asarray(lon, float) / deg).astype(int).astype(str))


def conformal_rank(n: int, alpha: float, delta: float | None = None) -> int:
    """delta=None: standard rank ceil((n+1)(1-alpha)), coverage >= 1-alpha on average.
    delta=0.05: smallest k with P(coverage >= 1-alpha) >= 1-delta (Vovk 2012)."""
    if delta is None:
        return int(np.ceil((n + 1) * (1 - alpha)))
    from scipy.stats import beta
    ks = np.arange(1, n + 1)
    ok = beta.sf(1 - alpha, ks, n + 1 - ks) >= 1 - delta
    return int(ks[ok][0]) if ok.any() else n + 1


def conformal_quantile(scores, alpha, delta=None) -> float:
    n = len(scores)
    k = conformal_rank(n, alpha, delta)
    return np.inf if k > n else float(np.sort(scores)[k - 1])


def predict_quantiles(bundle: dict, X: pd.DataFrame):
    """Lower/upper quantile predictions used by CQR (sorted so lo <= hi)."""
    a, b = bundle["qlo_model"].predict(X), bundle["qhi_model"].predict(X)
    return np.minimum(a, b), np.maximum(a, b)


def cqr_scores(bundle, X, y):
    lo, hi = predict_quantiles(bundle, X)
    return np.maximum(lo - y, y - hi)


def quantile_model(q):
    """Same booster settings as base_model, with quantile loss (no separate tuning)."""
    return HistGradientBoostingRegressor(
        loss="quantile", quantile=q, max_iter=900, learning_rate=0.04, max_leaf_nodes=24,
        min_samples_leaf=20, l2_regularization=1.0, early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=40, random_state=0)


def interval_score(y, lo, hi, alpha):
    return (hi - lo) + (2 / alpha) * ((lo - y) * (y < lo) + (y - hi) * (y > hi))


def predict_mean_sigma(bundle: dict, X: pd.DataFrame):
    mu = bundle["mean_model"].predict(X)
    sigma = np.exp(bundle["sigma_model"].predict(X))
    return mu, np.maximum(sigma, bundle["sigma_floor"])


def base_model():
    return HistGradientBoostingRegressor(
        max_iter=900, learning_rate=0.04, max_leaf_nodes=24, min_samples_leaf=20,
        l2_regularization=1.0, early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=40, random_state=0)


def sigma_model():
    return HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.04, max_leaf_nodes=12, min_samples_leaf=40,
        l2_regularization=2.0, random_state=1)


def block_split(blocks, rng, fracs=(0.6, 0.2, 0.2)):
    ub = rng.permutation(np.unique(blocks))
    n1 = int(round(fracs[0] * len(ub))); n2 = n1 + int(round(fracs[1] * len(ub)))
    return (np.isin(blocks, ub[:n1]), np.isin(blocks, ub[n1:n2]), np.isin(blocks, ub[n2:]))


def oof_residuals(X, y, groups, n_splits=5):
    oof = np.zeros(len(y))
    for tr, va in GroupKFold(n_splits=n_splits).split(X, y, groups):
        oof[va] = y[va] - clone(base_model()).fit(X.iloc[tr], y[tr]).predict(X.iloc[va])
    return oof


def diagnostics_figure(test_df, lo, hi, mu, rep_cov, group_cov, alpha, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.spines.top": False,
                         "axes.spines.right": False, "axes.edgecolor": "#c3c2b7"})
    BLUE, RED, INK = "#2a78d6", "#d03b3b", "#0b0b0b"
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.6), facecolor="#fcfcfb", gridspec_kw={"width_ratios": [1.5, 1, 1]})
    y = test_df[TARGET].to_numpy(); order = np.argsort(mu)
    sub = order[np.linspace(0, len(order) - 1, min(160, len(order))).astype(int)]
    cov = (y >= lo) & (y <= hi); xs = np.arange(len(sub)); ax = axs[0]
    ax.fill_between(xs, lo[sub], hi[sub], color=BLUE, alpha=0.18, lw=0, label=f"{1-alpha:.0%} conformal interval")
    ax.plot(xs, mu[sub], color=BLUE, lw=2, label="prediction")
    ax.scatter(xs[cov[sub]], y[sub][cov[sub]], s=10, color=INK, lw=0, label="observed, inside")
    ax.scatter(xs[~cov[sub]], y[sub][~cov[sub]], s=24, color=RED, marker="x", label="observed, outside")
    ax.set_xlabel("Test lakes (sorted by prediction)"); ax.set_ylabel("3-year log area change")
    ax.set_title("Held-out test blocks (2018→2021)", loc="left", color=INK); ax.legend(frameon=False, fontsize=8)
    ax = axs[1]; names, vals, ns = zip(*group_cov); yp = np.arange(len(names))
    ax.barh(yp, vals, color=BLUE, height=0.6); ax.axvline(1 - alpha, color="#52514e", ls="--", lw=1)
    for i, (v, n) in enumerate(zip(vals, ns)):
        ax.text(v + 0.005, i, f"{v:.1%} (n={n})", va="center", fontsize=8, color="#52514e")
    ax.set_yticks(yp, names); ax.set_xlim(0.6, 1.1); ax.invert_yaxis(); ax.set_xlabel("Coverage")
    ax.set_title("Coverage by group (dashed = target)", loc="left", color=INK)
    ax = axs[2]; ax.hist(rep_cov, bins=30, color=BLUE, alpha=0.85)
    ax.axvline(1 - alpha, color="#52514e", ls="--", lw=1); ax.axvline(np.mean(rep_cov), color=INK, lw=1.5)
    ax.set_xlabel("Test coverage"); ax.set_yticks([])
    ax.set_title(f"300 re-drawn calib/test splits\nmean {np.mean(rep_cov):.3f}", loc="left", color=INK)
    fig.tight_layout(); fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--alpha", type=float, default=0.10)
    ap.add_argument("--block-deg", type=float, default=0.5)
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--delta", default="0.05", help="number, or 'none' for standard split CP")
    args = ap.parse_known_args(argv)[0]
    delta = None if str(args.delta).lower() == "none" else float(args.delta)
    for d in ("models", "outputs", "figures"):
        os.makedirs(d, exist_ok=True)

    df = pd.read_csv("data/train_lakes.csv")
    X_all, y_all = make_X(df), df[TARGET].to_numpy()
    blocks = spatial_blocks(df.lat, df.lon, args.block_deg)
    rng = np.random.default_rng(args.seed)
    tr, ca, te = block_split(blocks, rng)
    print(f"Spatial blocks: {len(np.unique(blocks))} cells of {args.block_deg}° | "
          f"lakes train/calib/test = {tr.sum()}/{ca.sum()}/{te.sum()}")

    Xtr, ytr = X_all[tr], y_all[tr]
    mean_model = clone(base_model()).fit(Xtr, ytr)
    r_oof = oof_residuals(Xtr.reset_index(drop=True), ytr, blocks[tr])
    sig = clone(sigma_model()).fit(Xtr, np.log(np.abs(r_oof) + 1e-3))
    bundle = {"mean_model": mean_model, "sigma_model": sig,
              "sigma_floor": float(np.quantile(np.abs(r_oof), 0.05) + 1e-3),
              "qlo_model": clone(quantile_model(args.alpha / 2)).fit(Xtr, ytr),
              "qhi_model": clone(quantile_model(1 - args.alpha / 2)).fit(Xtr, ytr)}

    mu_c, sg_c = predict_mean_sigma(bundle, X_all[ca])
    s_norm = np.abs(y_all[ca] - mu_c) / sg_c
    g_cal = df[GROUP_COL].to_numpy()[ca]
    q_norm = conformal_quantile(s_norm, args.alpha, delta)
    q_group = {g: conformal_quantile(s_norm[g_cal == g], args.alpha, delta)
               for g in np.unique(g_cal) if (g_cal == g).sum() >= 30}

    # CQR calibration
    s_cqr = cqr_scores(bundle, X_all[ca], y_all[ca])
    q_cqr = conformal_quantile(s_cqr, args.alpha, delta)
    q_cqr_group = {g: conformal_quantile(s_cqr[g_cal == g], args.alpha, delta)
                   for g in np.unique(g_cal) if (g_cal == g).sum() >= 30}

    mu_t, sg_t = predict_mean_sigma(bundle, X_all[te]); y_t = y_all[te]
    lo, hi = mu_t - q_norm * sg_t, mu_t + q_norm * sg_t
    covered = (y_t >= lo) & (y_t <= hi)
    test_df = df[te].reset_index(drop=True)
    qg = test_df[GROUP_COL].map(q_group).fillna(q_norm).to_numpy()
    cov_mond = float(np.mean(np.abs(y_t - mu_t) <= qg * sg_t))
    groups = [(f"{g}", (test_df[GROUP_COL] == g).to_numpy()) for g in ["Glacier-fed", "Non Glacier-fed"]]
    groups += [("small lake < 0.02 km²", (test_df.area_t0_km2 < 0.02).to_numpy()),
               ("lake ≥ 0.02 km²", (test_df.area_t0_km2 >= 0.02).to_numpy())]
    group_cov = [(n, float(covered[m].mean()), int(m.sum())) for n, m in groups if m.sum() > 0]

    # method choice on held-out 2018-2021 test blocks (2021-2024 is never used here)
    g_te = test_df[GROUP_COL]
    qn = g_te.map(q_group).fillna(q_norm).to_numpy()
    qc = g_te.map(q_cqr_group).fillna(q_cqr).to_numpy()
    ql_t, qh_t = predict_quantiles(bundle, X_all[te])
    cand = {"normalized-mondrian": (mu_t - qn * sg_t, mu_t + qn * sg_t),
            "cqr-mondrian": (ql_t - qc, qh_t + qc)}
    comparison = {k: {"coverage": float(np.mean((y_t >= l) & (y_t <= h))), "mean_width": float(np.mean(h - l)),
                      "interval_score": float(np.mean(interval_score(y_t, l, h, args.alpha)))}
                  for k, (l, h) in cand.items()}
    eligible = {k: v for k, v in comparison.items() if v["coverage"] >= 0.88} or comparison
    preferred = min(eligible, key=lambda k: eligible[k]["interval_score"])

    pool = ca | te
    mu_p, sg_p = predict_mean_sigma(bundle, X_all[pool])
    s_p, b_p = np.abs(y_all[pool] - mu_p) / sg_p, blocks[pool]
    ub = np.unique(b_p); rep_cov = []
    for _ in range(300):
        mc = np.isin(b_p, rng.permutation(ub)[: len(ub) // 2])
        rep_cov.append(float(np.mean(s_p[~mc] <= conformal_quantile(s_p[mc], args.alpha, delta))))
    rep_cov = np.array(rep_cov)

    bundle.update({"q_cqr": q_cqr, "q_cqr_group": q_cqr_group, "cal_scores_cqr": s_cqr,
                   "preferred_method": preferred, "step2_method_comparison": comparison})
    bundle.update({"alpha": args.alpha, "delta": delta, "q_norm": q_norm, "q_group": q_group,
                   "cal_scores_norm": s_norm, "cal_groups": g_cal, "X_cal": X_all[ca].reset_index(drop=True),
                   "feature_columns": FEATURES, "group_col": GROUP_COL, "block_deg": args.block_deg,
                   "train_feature_range": {c: (float(X_all[c].quantile(0.005)), float(X_all[c].quantile(0.995)))
                                           for c in FEATURES}})
    joblib.dump(bundle, "models/glo_cp_bundle.joblib")
    r2, mae = r2_score(y_t, mu_t), mean_absolute_error(y_t, mu_t)
    metrics = {"alpha": args.alpha, "delta": delta, "n_train": int(tr.sum()), "n_calibration": int(ca.sum()),
               "n_test": int(te.sum()), "test_R2": round(r2, 4), "test_MAE": round(mae, 4),
               "coverage_normalized": round(float(covered.mean()), 4), "coverage_mondrian": round(cov_mond, 4),
               "mean_width": round(float(np.mean(hi - lo)), 4),
               "repeated_split_coverage_mean": round(float(rep_cov.mean()), 4),
               "repeated_split_coverage_p05": round(float(np.quantile(rep_cov, 0.05)), 4),
               "q_pooled": q_norm, "q_by_group": q_group, "q_cqr": q_cqr, "q_cqr_by_group": q_cqr_group,
               "method_comparison_on_2018_2021_test_blocks": comparison, "preferred_method": preferred,
               "group_coverage": {n: round(v, 4) for n, v, _ in group_cov}}
    json.dump(metrics, open("outputs/glo_step2_metrics.json", "w"), indent=2)
    diagnostics_figure(test_df, lo, hi, mu_t, rep_cov, group_cov, args.alpha, "figures/glo_step2_training_diagnostics.png")

    print("\nGLO STEP 2 complete - trained on 2018->2021 only")
    print(f"  Test R²                      : {r2:.3f}")
    print(f"  Test MAE (3-yr log change)   : {mae:.3f}")
    print(f"  Coverage, normalized split CP: {covered.mean():.3f} (target ≥ {1-args.alpha:.2f}), mean width {np.mean(hi-lo):.3f}")
    print(f"  Coverage, per connectivity   : {cov_mond:.3f}")
    print(f"  300 re-drawn splits          : mean {rep_cov.mean():.3f}, 5th pct {np.quantile(rep_cov, 0.05):.3f}")
    for n, v, k in group_cov:
        print(f"    {n:<24s} {v:.3f} (n={k})")
    print("  Method comparison on held-out 2018->2021 blocks (chosen without 2021-2024):")
    for k, v in comparison.items():
        print(f"    {k:<20s} coverage {v['coverage']:.3f}  width {v['mean_width']:.3f}  "
              f"interval score {v['interval_score']:.3f}{'  <- preferred' if k == preferred else ''}")


if __name__ == "__main__":
    main()

Spatial blocks: 49 cells of 0.5° | lakes train/calib/test = 1395/277/602

GLO STEP 2 complete - trained on 2018->2021 only
  Test R²                      : 0.314
  Test MAE (3-yr log change)   : 0.223
  Coverage, normalized split CP: 0.932 (target ≥ 0.90), mean width 1.287
  Coverage, per connectivity   : 0.958
  300 re-drawn splits          : mean 0.921, 5th pct 0.888
    Glacier-fed              0.920 (n=350)
    Non Glacier-fed          0.948 (n=252)
    small lake < 0.02 km²    0.924 (n=290)
    lake ≥ 0.02 km²          0.939 (n=312)
  Method comparison on held-out 2018->2021 blocks (chosen without 2021-2024):
    normalized-mondrian  coverage 0.958  width 1.561  interval score 1.742
    cqr-mondrian         coverage 0.939  width 1.322  interval score 1.660  <- preferred


In [5]:
"""
GLO STEP 3 - Analyse 2021 -> 2024 with the frozen 2018 -> 2021 model
====================================================================

Same as the synthetic Step 3, but the "independent dataset" is the REAL future:
the model and its calibration come only from 2018 -> 2021 (Step 2) and are never
retrained. The 2024 lake areas (answer key) are read only in the final
validation block.

Input : models/glo_cp_bundle.joblib, data/analysis_features.csv,
        data/analysis_answer_key.csv (validation only), data/generation_meta.json
Output: outputs/glo_analysis_predictions.csv, outputs/glo_high_risk_zones.csv,
        outputs/glo_step3_validation.json,
        figures/glo_step3_spatial_uncertainty_maps.png, figures/glo_step3_validation.png

Conformal intervals (choose with --method; all use Step 2's calibration only):
  standard           normalized split CP, one q from Step 2
  mondrian           separate q for glacier-fed and non glacier-fed lakes
  mondrian-weighted  mondrian + covariate-shift weights (Tibshirani et al. 2019)
  cqr                conformalized quantile regression, one q (lopsided intervals)
  cqr-mondrian       CQR with a separate q per connectivity group
  auto (default)     the method Step 2 preferred on its held-out 2018-2021 blocks
A fourth check, "recalibrated", is validation only: it keeps the frozen model
but calibrates on half of the 2021->2024 lakes (by 0.5° blocks) and tests on
the other half, 200 times. It shows what local, recent recalibration achieves.

GLOF susceptibility (GLO has no dam type or slopes, so a growth-based proxy):
  S = 0.45 s_size(area after 3 yr) + 0.30 s_growth + 0.25 s_contact(glacier-fed)
evaluated at the lower bound, point and upper bound of each lake's interval.
Classes: High (confident) S_lo >= T; High (possible) S_hi >= T > S_lo;
Moderate S_hi >= T_mod; Low otherwise.

Run:  python glo_step3_analyse_2021_2024.py [--method mondrian] [--risk-threshold 0.6]
In a notebook: run the Step 1 and Step 2 cells first (or save them as .py files
next to the notebook), then paste this cell.
"""
from __future__ import annotations

import argparse
import importlib
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def _from_earlier_step(module, names):
    """Import from a saved .py file, or from functions defined by an earlier notebook cell."""
    try:
        mod = importlib.import_module(module)
        return [getattr(mod, n) for n in names]
    except ModuleNotFoundError as err:
        if err.name != module:
            raise
    import __main__
    session = vars(__main__)
    if any(n not in session for n in names):
        raise ModuleNotFoundError(f"Can't find {module}. Save it as {module}.py or run that step's cell first.",
                                  name=module)
    return [session[n] for n in names]


(FEATURES, conformal_quantile, make_X, predict_mean_sigma, predict_quantiles, spatial_blocks) = _from_earlier_step(
    "glo_step2_train_calibrate",
    ["FEATURES", "conformal_quantile", "make_X", "predict_mean_sigma", "predict_quantiles", "spatial_blocks"])

METHODS = ["standard", "mondrian", "mondrian-weighted", "cqr", "cqr-mondrian"]
PREFERRED_TO_METHOD = {"normalized-mondrian": "mondrian", "cqr-mondrian": "cqr-mondrian"}
CONTACT_W = {1.0: 1.0, 0.0: 0.25}


# ---------------------------------------------------------------------------
def shift_weights(X_cal, X_new, clip=(0.1, 10.0)):
    Z = pd.concat([X_cal, X_new], ignore_index=True).fillna(0)
    lab = np.r_[np.zeros(len(X_cal)), np.ones(len(X_new))]
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=2000)).fit(Z, lab)
    p_cal = np.clip(clf.predict_proba(X_cal.fillna(0))[:, 1], 1e-4, 1 - 1e-4)
    p_new = np.clip(clf.predict_proba(X_new.fillna(0))[:, 1], 1e-4, 1 - 1e-4)
    r = len(X_cal) / len(X_new)
    return np.clip(p_cal / (1 - p_cal) * r, *clip), np.clip(p_new / (1 - p_new) * r, *clip)


def weighted_q(scores, w_cal, w_new, alpha):
    o = np.argsort(scores); s, cw = scores[o], np.cumsum(w_cal[o])
    idx = np.searchsorted(cw, (1 - alpha) * (cw[-1] + w_new), side="left")
    q = np.full(len(w_new), np.inf); ok = idx < len(s); q[ok] = s[idx[ok]]
    return q


def interval_pipeline(bundle, df):
    """Frozen prediction + interval (lo, hi) for every method.
    Normalized methods: mu ± q·sigma.  CQR methods: [q_lo - q, q_hi + q]."""
    X = make_X(df)[bundle["feature_columns"]]
    mu, sg = predict_mean_sigma(bundle, X)
    sc, a, d = bundle["cal_scores_norm"], bundle["alpha"], bundle.get("delta")
    g_cal, g_new = bundle["cal_groups"], df[bundle["group_col"]].to_numpy()
    w_cal, w_new = shift_weights(bundle["X_cal"], X)
    q = {"standard": np.full(len(df), bundle["q_norm"]),
         "mondrian": pd.Series(g_new).map(bundle["q_group"]).fillna(bundle["q_norm"]).to_numpy()}
    qw = weighted_q(sc, w_cal, w_new, a)
    for g in np.unique(g_new):
        mc, mn = g_cal == g, g_new == g
        if mc.sum() >= 30:
            qw[mn] = weighted_q(sc[mc], w_cal[mc], w_new[mn], a)
    q["mondrian-weighted"] = np.minimum(qw, 3 * sc.max())
    iv = {k: (mu - v * sg, mu + v * sg) for k, v in q.items()}
    ql, qh = predict_quantiles(bundle, X)
    qc = {"cqr": np.full(len(df), bundle["q_cqr"]),
          "cqr-mondrian": pd.Series(g_new).map(bundle["q_cqr_group"]).fillna(bundle["q_cqr"]).to_numpy()}
    for k, v in qc.items():
        iv[k] = (ql - v, qh + v)
    return mu, sg, iv, (ql, qh), w_cal, w_new


def susceptibility(area_t0, y, glacier_fed, growth_full=0.3):
    a1 = area_t0 * np.exp(y)
    s_size = np.clip(np.log10(a1 / 0.02) / np.log10(50), 0, 1)
    s_growth = np.clip(y / growth_full, 0, 1)
    s_contact = pd.Series(glacier_fed).map(CONTACT_W).fillna(0.25).to_numpy()
    return 0.45 * s_size + 0.30 * s_growth + 0.25 * s_contact


def risk_class(s_lo, s_hi, t, tm):
    return np.select([s_lo >= t, s_hi >= t, s_hi >= tm], ["High (confident)", "High (possible)", "Moderate"], "Low")


def kernel_grid(lat, lon, values, bbox, res=0.02, bw=0.06, min_density=0.6):
    la = np.arange(bbox[0], bbox[1] + res, res); lo = np.arange(bbox[2], bbox[3] + res, res)
    G_lo, G_la = np.meshgrid(lo, la); cl = np.cos(np.deg2rad(np.mean(bbox[:2])))
    Z = np.full(G_la.size, np.nan); dens = np.zeros(G_la.size)
    gl, gn = G_la.ravel(), G_lo.ravel()
    for s in range(0, G_la.size, 4000):
        d2 = (gl[s:s+4000, None] - lat[None]) ** 2 + ((gn[s:s+4000, None] - lon[None]) * cl) ** 2
        K = np.exp(-0.5 * d2 / bw ** 2); dn = K.sum(1)
        dens[s:s+4000] = dn; Z[s:s+4000] = (K @ values) / np.maximum(dn, 1e-12)
    Z[dens < min_density] = np.nan
    return G_lo, G_la, Z.reshape(G_la.shape)


def _style():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.spines.top": False,
                         "axes.spines.right": False, "axes.edgecolor": "#c3c2b7"})
    return plt


CLASS_STYLE = {"High (confident)": ("#d03b3b", "^", 30), "High (possible)": ("#ec835a", "D", 16),
               "Moderate": ("#fab219", "o", 7), "Low": ("#c3c2b7", "o", 4)}


def map_figure(df, bbox, zones, path):
    plt = _style()
    from matplotlib.colors import LinearSegmentedColormap as LSC
    div = LSC.from_list("d", ["#104281", "#3987e5", "#f0efec", "#e66767", "#9e2222"])
    seqb = LSC.from_list("b", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
    seqo = LSC.from_list("o", ["#fde3d6", "#f5ab88", "#eb6834", "#b8461c", "#6e2a10"])
    lat, lon = df.lat.to_numpy(), df.lon.to_numpy()
    fig, axs = plt.subplots(2, 2, figsize=(15, 8.5), facecolor="#fcfcfb")
    for ax, (col, title, cmap) in zip(axs.ravel()[:3], [
            ("pct_change_pred", "(a) Predicted lake-area change 2021→2024 (%)", div),
            ("interval_width", "(b) Spatial uncertainty: 90 % interval width (log ratio)", seqb),
            ("risk_hi", "(c) GLOF susceptibility, upper bound of 90 % interval", seqo)]):
        v = df[col].to_numpy()
        if col == "pct_change_pred":
            v = np.clip(v, -60, 60)
        Gx, Gy, Z = kernel_grid(lat, lon, v, bbox)
        if col == "pct_change_pred":
            m = np.nanmax(np.abs(np.nanpercentile(Z, [2, 98]))); vmin, vmax = -m, m
        elif col == "risk_hi":
            vmin, vmax = 0, 0.8
        else:
            vmin, vmax = np.nanpercentile(Z, [2, 98])
        pc = ax.pcolormesh(Gx, Gy, Z, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
        ax.scatter(lon, lat, s=1, c="#52514e", alpha=0.3, lw=0); fig.colorbar(pc, ax=ax, fraction=0.03, pad=0.01)
        ax.set_title(title, loc="left", fontsize=10)
    ax = axs[1, 1]
    for cl in ["Low", "Moderate", "High (possible)", "High (confident)"]:
        m = (df.risk_class == cl).to_numpy(); c, mk, s = CLASS_STYLE[cl]
        ax.scatter(lon[m], lat[m], s=s, c=c, marker=mk, lw=0.3, edgecolors="#fcfcfb", label=f"{cl} (n={m.sum()})", zorder=3)
    for _, z in zones.head(5).iterrows():
        ax.annotate(f"Z{int(z['rank'])}", (z.lon_center, z.lat_center), xytext=(5, 5), textcoords="offset points",
                    fontsize=9, weight="bold", zorder=4)
    ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=4, markerscale=1.3)
    ax.set_title("(d) Lake susceptibility classes and top-5 high-risk zones", loc="left", fontsize=10)
    for ax in axs.ravel():
        ax.set_xlim(bbox[2], bbox[3]); ax.set_ylim(bbox[0], bbox[1])
        ax.set_aspect(1 / np.cos(np.deg2rad(np.mean(bbox[:2])))); ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
    fig.suptitle("GLO Nepal lakes, 2021→2024 - frozen 2018→2021 model + split conformal prediction", x=0.01, ha="left", fontsize=12)
    fig.tight_layout(); fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)


def validation_figure(res, df, alpha, t_high, path):
    plt = _style(); BLUE, GRAY, INK = "#2a78d6", "#898781", "#0b0b0b"
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.6), facecolor="#fcfcfb", gridspec_kw={"width_ratios": [1.1, 1, 1.3]})
    ax = axs[0]
    rk = list(res["recalibrated_all"].keys())
    names = METHODS + [f"{k}, recalibrated" for k in rk]
    vals = [res["coverage"][m] for m in METHODS] + [res["recalibrated_all"][k]["coverage_mean"] for k in rk]
    widths = [res["mean_width"][m] for m in METHODS] + [res["recalibrated_all"][k]["mean_width"] for k in rk]
    cols = [BLUE if n == res["method_used"] else GRAY for n in METHODS] + ["#1baf7a"] * len(rk)
    ax.barh(range(len(names)), vals, color=cols, height=0.55)
    for i, (v, w) in enumerate(zip(vals, widths)):
        ax.text(v + 0.004, i, f"{v:.1%}  width {w:.2f}", va="center", fontsize=8, color="#52514e")
    ax.axvline(1 - alpha, color="#52514e", ls="--", lw=1)
    ax.set_yticks(range(len(names)), names); ax.set_xlim(0.75, 1.08); ax.invert_yaxis()
    ax.set_title("Coverage on 2021→2024 (dashed = 90 % target;\nblue = method used, green = recalibrated)", loc="left", color=INK)
    ax = axs[1]; g = res["coverage_by_group"]
    ax.barh(range(len(g)), list(g.values()), color=BLUE, height=0.55)
    for i, v in enumerate(g.values()):
        ax.text(v + 0.004, i, f"{v:.1%}", va="center", fontsize=8, color="#52514e")
    ax.axvline(1 - alpha, color="#52514e", ls="--", lw=1)
    ax.set_yticks(range(len(g)), list(g.keys())); ax.set_xlim(0.6, 1.08); ax.invert_yaxis()
    ax.set_title(f"Coverage by group ({res['method_used']})", loc="left", color=INK)
    ax = axs[2]; top = df.dropna(subset=["risk_true"]).sort_values("risk_point", ascending=False).head(60).reset_index(drop=True)
    x = np.arange(len(top))
    ax.vlines(x, top.risk_lo, top.risk_hi, color=BLUE, lw=2, alpha=0.55, label="90 % risk interval")
    ax.scatter(x, top.risk_point, s=12, color=BLUE, zorder=3, label="predicted")
    ins = (top.risk_true >= top.risk_lo - 1e-12) & (top.risk_true <= top.risk_hi + 1e-12)
    ax.scatter(x[ins], top.risk_true[ins], s=16, color=INK, marker="_", lw=2, zorder=4, label="observed 2024")
    ax.scatter(x[~ins], top.risk_true[~ins], s=26, color="#d03b3b", marker="x", zorder=4, label="observed, outside")
    ax.axhline(t_high, color="#52514e", ls="--", lw=1); ax.set_xlabel("60 highest-ranked lakes")
    ax.set_ylabel("GLOF susceptibility index"); ax.legend(frameon=False, fontsize=8, loc="lower left")
    ax.set_title("Risk intervals vs observed 2024", loc="left", color=INK)
    fig.tight_layout(); fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)


# ---------------------------------------------------------------------------
def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--method", choices=METHODS + ["auto"], default="auto",
                    help="auto = the method Step 2 preferred on its own held-out 2018-2021 blocks")
    ap.add_argument("--risk-threshold", type=float, default=0.60)
    ap.add_argument("--moderate-threshold", type=float, default=0.40)
    ap.add_argument("--zone-deg", type=float, default=0.25)
    ap.add_argument("--recal-repeats", type=int, default=200)
    args = ap.parse_known_args(argv)[0]
    os.makedirs("outputs", exist_ok=True); os.makedirs("figures", exist_ok=True)

    bundle = joblib.load("models/glo_cp_bundle.joblib")
    meta = json.load(open("data/generation_meta.json"))
    df = pd.read_csv("data/analysis_features.csv")
    assert "log_area_change" not in df.columns
    alpha = bundle["alpha"]
    if args.method == "auto":
        args.method = PREFERRED_TO_METHOD.get(bundle.get("preferred_method"), "mondrian")

    # 1-2. frozen predictions and intervals
    mu, sg, iv, (ql, qh), w_cal, w_new = interval_pipeline(bundle, df)
    lo, hi = iv[args.method]
    rng_ = bundle["train_feature_range"]
    n_out = sum(((df[c] < rng_[c][0]) | (df[c] > rng_[c][1])).astype(int) for c in FEATURES if c in df)
    gf = df.glacier_fed.to_numpy(); a0 = df.area_t0_km2.to_numpy()
    s_pt, s_lo, s_hi = susceptibility(a0, mu, gf), susceptibility(a0, lo, gf), susceptibility(a0, hi, gf)
    cls = risk_class(s_lo, s_hi, args.risk_threshold, args.moderate_threshold)

    out = df[["lake_id", "lat", "lon", "basin", "country", "connectivity", "area_t0_km2"]].copy()
    out["log_change_pred"], out["log_change_lo"], out["log_change_hi"] = mu, lo, hi
    out["interval_width"] = hi - lo
    out["area_2024_pred_km2"], out["area_2024_lo_km2"], out["area_2024_hi_km2"] = a0 * np.exp(mu), a0 * np.exp(lo), a0 * np.exp(hi)
    out["pct_change_pred"], out["pct_change_lo"], out["pct_change_hi"] = [100 * (np.exp(v) - 1) for v in (mu, lo, hi)]
    out["trend_call"] = np.select([lo > 0, hi < 0], ["growing", "shrinking"], "uncertain")
    out["shift_weight"] = w_new; out["n_features_outside_training_range"] = n_out
    out["risk_point"], out["risk_lo"], out["risk_hi"], out["risk_class"] = s_pt, s_lo, s_hi, cls
    out.to_csv("outputs/glo_analysis_predictions.csv", index=False)

    zd = args.zone_deg
    out["zone"] = np.floor(out.lat / zd).astype(int).astype(str) + "_" + np.floor(out.lon / zd).astype(int).astype(str)
    z = out.groupby("zone").agg(n_lakes=("lake_id", "size"),
                                n_high_confident=("risk_class", lambda s: (s == "High (confident)").sum()),
                                n_high_possible=("risk_class", lambda s: (s == "High (possible)").sum()),
                                max_risk_hi=("risk_hi", "max"), mean_interval_width=("interval_width", "mean"),
                                lat_center=("lat", "mean"), lon_center=("lon", "mean"), basin=("basin", "first"))
    z["zone_score"] = z.n_high_confident + 0.5 * z.n_high_possible + z.max_risk_hi
    z = z[(z.n_high_confident + z.n_high_possible) > 0].sort_values("zone_score", ascending=False)
    z.insert(0, "rank", np.arange(1, len(z) + 1)); z.round(4).to_csv("outputs/glo_high_risk_zones.csv")
    b = meta["bbox"]; bbox = [b[0] - 0.1, b[1] + 0.1, b[2] - 0.1, b[3] + 0.1]
    map_figure(out, bbox, z, "figures/glo_step3_spatial_uncertainty_maps.png")

    # 3. validation against the 2024 answer key (lakes still mapped in 2024)
    key = pd.read_csv("data/analysis_answer_key.csv").set_index("lake_id").loc[df.lake_id]
    y = key.log_area_change.to_numpy(); ok = np.isfinite(y)
    s_true = np.full(len(df), np.nan); s_true[ok] = susceptibility(a0[ok], y[ok], gf[ok])
    out["risk_true"] = s_true
    sc = np.abs(y - mu) / sg
    cov = {k: float(np.mean(((y >= l) & (y <= h))[ok])) for k, (l, h) in iv.items()}
    wid = {k: float(np.mean(h - l)) for k, (l, h) in iv.items()}
    inside = (y >= lo) & (y <= hi)
    groups = {"Glacier-fed": (gf == 1) & ok, "Non glacier-fed": (gf == 0) & ok,
              "small < 0.02 km²": (a0 < 0.02) & ok, "≥ 0.02 km²": (a0 >= 0.02) & ok,
              "≥1 input outside training range": (n_out >= 1) & ok}
    for bn in sorted(df.basin.dropna().unique()):
        groups[f"basin: {bn}"] = (df.basin == bn).to_numpy() & ok
    cov_g = {k: float(inside[m].mean()) for k, m in groups.items() if m.sum() >= 20}

    # recalibrated: frozen model, calibrate on half the 2021->2024 blocks, test on the other half
    blk = spatial_blocks(df.lat, df.lon, bundle["block_deg"])[ok]
    gro = df[bundle["group_col"]].to_numpy()[ok]; ub = np.unique(blk)
    fam = {"normalized-mondrian": (sc[ok], 2 * sg[ok], np.zeros(ok.sum())),
           "cqr-mondrian": (np.maximum(ql - y, y - qh)[ok], 2 * np.ones(ok.sum()), (qh - ql)[ok])}
    recal = {}
    for name, (sco, scale, base_w) in fam.items():
        rng = np.random.default_rng(11); rc, rw = [], []
        for _ in range(args.recal_repeats):
            mc = np.isin(blk, rng.permutation(ub)[: len(ub) // 2])
            qp = conformal_quantile(sco[mc], alpha, bundle.get("delta"))
            qg = {g: conformal_quantile(sco[mc & (gro == g)], alpha, bundle.get("delta"))
                  for g in np.unique(gro) if (mc & (gro == g)).sum() >= 30}
            qq = pd.Series(gro[~mc]).map(qg).fillna(qp).to_numpy()
            rc.append(float(np.mean(sco[~mc] <= qq)))
            rw.append(float(np.mean(base_w[~mc] + qq * scale[~mc])))
        recal[name] = {"coverage_mean": float(np.mean(rc)), "coverage_p05": float(np.quantile(rc, 0.05)),
                       "mean_width": float(np.mean(rw)), "repeats": args.recal_repeats}
    used_family = "cqr-mondrian" if args.method.startswith("cqr") else "normalized-mondrian"
    th = np.where(ok, s_true >= args.risk_threshold, False); fl = np.char.startswith(cls.astype(str), "High")
    res = {"n_lakes_2021": int(len(df)), "n_with_2024_answer": int(ok.sum()), "alpha": alpha, "method_used": args.method,
           "R2": float(r2_score(y[ok], mu[ok])), "coverage": cov, "mean_width": wid, "coverage_by_group": cov_g,
           "recalibrated": recal[used_family], "recalibrated_all": recal,
           "step2_preferred_method": bundle.get("preferred_method"),
           "step2_method_comparison": bundle.get("step2_method_comparison"),
           "risk_interval_coverage": float(np.mean((s_true[ok] >= s_lo[ok] - 1e-12) & (s_true[ok] <= s_hi[ok] + 1e-12))),
           "truly_high": int(th.sum()),
           "recall_High_flags": float(fl[th].mean()) if th.any() else None,
           "recall_point_only": float((s_pt >= args.risk_threshold)[th].mean()) if th.any() else None,
           "precision_High_confident": float(th[cls == "High (confident)"].mean()) if (cls == "High (confident)").any() else None,
           "risk_class_counts": pd.Series(cls).value_counts().to_dict(),
           "trend_calls": out.trend_call.value_counts().to_dict(), "high_risk_zones": int(len(z)),
           "effective_sample_size_of_weights": float(w_cal.sum() ** 2 / (w_cal ** 2).sum())}
    json.dump(res, open("outputs/glo_step3_validation.json", "w"), indent=2, default=float)
    out.to_csv("outputs/glo_analysis_predictions.csv", index=False)
    validation_figure(res, out, alpha, args.risk_threshold, "figures/glo_step3_validation.png")

    print("GLO STEP 3 complete - 2021->2024 analysed with the frozen 2018->2021 model")
    print(f"  Lakes in 2021: {len(df)}, still mapped in 2024 (answer available): {ok.sum()}")
    print(f"  R² on 2021->2024: {res['R2']:.3f}")
    for k in METHODS:
        print(f"  Coverage {k:<18s}: {cov[k]:.3f}  (mean width {wid[k]:.3f}){'  <- used' if k == args.method else ''}")
    for k, r in recal.items():
        print(f"  {k} recalibrated on half of 2021->2024 ({r['repeats']} splits): mean {r['coverage_mean']:.3f}, "
              f"5th pct {r['coverage_p05']:.3f}, width {r['mean_width']:.3f}")
    for k, v in cov_g.items():
        print(f"    {k:<34s} {v:.3f} (n={groups[k].sum()})")
    print(f"  Risk classes: {res['risk_class_counts']} | trend calls: {res['trend_calls']}")
    print(f"  Truly high: {res['truly_high']} | caught by High flags: {res['recall_High_flags']} | "
          f"point only: {res['recall_point_only']} | precision High (confident): {res['precision_High_confident']}")
    print(z.head(5)[["rank", "basin", "n_lakes", "n_high_confident", "n_high_possible", "max_risk_hi",
                     "lat_center", "lon_center"]].round(3).to_string())


if __name__ == "__main__":
    main()

GLO STEP 3 complete - 2021->2024 analysed with the frozen 2018->2021 model
  Lakes in 2021: 2755, still mapped in 2024 (answer available): 2429
  R² on 2021->2024: 0.267
  Coverage standard          : 0.942  (mean width 1.518)
  Coverage mondrian          : 0.963  (mean width 1.895)
  Coverage mondrian-weighted : 0.937  (mean width 1.542)
  Coverage cqr               : 0.948  (mean width 1.385)
  Coverage cqr-mondrian      : 0.958  (mean width 1.473)  <- used
  normalized-mondrian recalibrated on half of 2021->2024 (200 splits): mean 0.919, 5th pct 0.890, width 1.213
  cqr-mondrian recalibrated on half of 2021->2024 (200 splits): mean 0.918, 5th pct 0.884, width 1.044
    Glacier-fed                        0.961 (n=1534)
    Non glacier-fed                    0.954 (n=895)
    small < 0.02 km²                   0.941 (n=1252)
    ≥ 0.02 km²                         0.977 (n=1177)
    ≥1 input outside training range    0.879 (n=132)
    basin: Gandaki                     0.959 (n=316)
  

In [6]:
"""
GLO STEP 4 - List the lakes flagged High and check them against 2024
====================================================================

Reads Step 3's predictions and the hidden 2024 answer key, and writes one table
of every lake flagged "High (confident)" or "High (possible)", with:
  location, basin, size in 2021, predicted 2024 range, observed 2024 area,
  susceptibility interval, and whether the lake was truly high-susceptibility in 2024.
Well-known hazardous lakes are labelled when a GLO lake lies within 1.5 km of
their published coordinates.

Input : outputs/glo_analysis_predictions.csv (Step 3), data/analysis_answer_key.csv
Output: outputs/glo_flagged_lakes.csv

Run:  python glo_step4_list_flagged_lakes.py [--risk-threshold 0.6] [--top 30]
In a notebook: run Steps 1-3 first, then paste this cell.
"""
from __future__ import annotations

import argparse
import os

import numpy as np
import pandas as pd

# Approximate published coordinates (lat, lon) of well-known hazardous lakes in the study area
KNOWN_LAKES = {
    "Imja Tsho": (27.898, 86.925),
    "Tsho Rolpa": (27.862, 86.475),
    "Thulagi": (28.487, 84.483),
    "Lower Barun": (27.795, 87.098),
    "Dig Tsho": (27.874, 86.585),
}
COLUMNS = ["lake_id", "known_name", "risk_class", "basin", "country", "connectivity", "lat", "lon",
           "area_t0_km2", "pct_change_pred", "pct_change_lo", "pct_change_hi",
           "area_2024_lo_km2", "area_2024_hi_km2", "area_2024_observed_km2", "inside_area_range",
           "risk_lo", "risk_point", "risk_hi", "risk_true", "truly_high_2024"]


def label_known_lakes(p: pd.DataFrame, max_km: float = 1.5) -> pd.Series:
    names = pd.Series("", index=p.index)
    for name, (la, lo) in KNOWN_LAKES.items():
        d_km = np.hypot(p.lat - la, (p.lon - lo) * np.cos(np.deg2rad(la))) * 111.0
        i = d_km.idxmin()
        if d_km[i] <= max_km:
            names[i] = name
    return names


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--risk-threshold", type=float, default=0.60)
    ap.add_argument("--top", type=int, default=30, help="how many confident lakes to print")
    args = ap.parse_known_args(argv)[0]

    p = pd.read_csv("outputs/glo_analysis_predictions.csv")
    key = pd.read_csv("data/analysis_answer_key.csv").set_index("lake_id")
    p["area_2024_observed_km2"] = p.lake_id.map(key.area_t1_km2)
    p["inside_area_range"] = np.where(p.area_2024_observed_km2.notna(),
                                      (p.area_2024_observed_km2 >= p.area_2024_lo_km2) &
                                      (p.area_2024_observed_km2 <= p.area_2024_hi_km2), np.nan)
    p["truly_high_2024"] = np.where(p.risk_true.notna(), p.risk_true >= args.risk_threshold, np.nan)
    p["known_name"] = label_known_lakes(p)

    flagged = p[p.risk_class.str.startswith("High")].copy()
    flagged["_order"] = (flagged.risk_class != "High (confident)").astype(int)
    flagged = flagged.sort_values(["_order", "risk_lo", "risk_hi"], ascending=[True, False, False])
    os.makedirs("outputs", exist_ok=True)
    flagged[COLUMNS].round(4).to_csv("outputs/glo_flagged_lakes.csv", index=False)

    conf = flagged[flagged.risk_class == "High (confident)"]
    poss = flagged[flagged.risk_class == "High (possible)"]
    truly = p.truly_high_2024 == 1
    print("GLO STEP 4 complete - flagged lakes written to outputs/glo_flagged_lakes.csv")
    print(f"  Lakes analysed: {len(p)} | flagged High: {len(flagged)} "
          f"(confident {len(conf)}, possible {len(poss)})")
    print(f"  Truly high in 2024: {int(truly.sum())} | caught by High flags: "
          f"{int((truly & p.risk_class.str.startswith('High')).sum())}")
    for name, g in [("High (confident)", conf), ("High (possible)", poss)]:
        hit = g.truly_high_2024.sum()
        print(f"  {name:<17s}: {len(g):4d} lakes, truly high in 2024: {int(hit)} ({hit / max(len(g), 1):.0%}); "
              f"by basin {g.basin.value_counts().to_dict()}")
    print("  Well-known lakes:", {r.known_name: r.risk_class for r in p[p.known_name != ""].itertuples()})
    pd.set_option("display.width", 220)
    print(conf.head(args.top)[["lake_id", "known_name", "basin", "country", "area_t0_km2", "area_2024_lo_km2",
                               "area_2024_hi_km2", "area_2024_observed_km2", "risk_lo", "truly_high_2024"]]
          .round(3).to_string(index=False))


if __name__ == "__main__":
    main()

GLO STEP 4 complete - flagged lakes written to outputs/glo_flagged_lakes.csv
  Lakes analysed: 2755 | flagged High: 1029 (confident 30, possible 999)
  Truly high in 2024: 153 | caught by High flags: 150
  High (confident) :   30 lakes, truly high in 2024: 30 (100%); by basin {'Koshi': 29, 'Gandaki': 1}
  High (possible)  :  999 lakes, truly high in 2024: 120 (12%); by basin {'Koshi': 659, 'Karnali': 208, 'Gandaki': 132}
  Well-known lakes: {'Thulagi': 'High (possible)', 'Tsho Rolpa': 'High (confident)', 'Dig Tsho': 'High (possible)', 'Imja Tsho': 'High (confident)', 'Lower Barun': 'High (confident)'}
              lake_id  known_name   basin country  area_t0_km2  area_2024_lo_km2  area_2024_hi_km2  area_2024_observed_km2  risk_lo  truly_high_2024
GLO_83.85335_28.69074             Gandaki   Nepal        3.569             1.819             6.246                   3.598    0.700              1.0
GLO_85.84205_28.32067               Koshi   China        5.438             3.560             